In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

In [12]:

transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))

])
trainset=torchvision.datasets.CIFAR10(root='./data',train=True,download=True,transform=transform)
trainloader=torch.utils.data.DataLoader(trainset,batch_size=64,shuffle=True)

testset=torchvision.datasets.CIFAR10(root='/data',train=False,download=True,transform=transform)
testLoader=torch.utils.data.DataLoader(testset,batch_size=64,shuffle=False)


classes= ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']



100%|██████████| 170M/170M [21:49<00:00, 130kB/s]


In [19]:
class CNN(nn.Module):
  def __init__(self):
     super().__init__()
     self.conv1=nn.Conv2d(3,32,kernel_size=3,padding=1)
     self.conv2=nn.Conv2d(32,64,kernel_size=3,padding=1)
     self.pool=nn.MaxPool2d(2,2)
     self.fc1=nn.Linear(64*8*8,128)
     self.fc2=nn.Linear(128,10)
     self.relu=nn.ReLU()

  def forward(self,x):
      x = self.pool(self.relu(self.conv1(x)))
      x = self.pool(self.relu(self.conv2(x)))
      x = x.view(-1, 64 * 8 * 8)
      x = self.relu(self.fc1(x))
      x = self.fc2(x)
      return x

net=CNN()

In [20]:
crit=nn.CrossEntropyLoss()
optimizer=optim.Adam(net.parameters(),lr=0.001)


In [24]:
epochs = 20
for epochs in range(epochs):
  rl=0
  for images,labels in trainloader:
    images,labels=images,labels
    optimizer.zero_grad()
    outputs=net(images)
    loss = crit(outputs, labels)
    loss.backward()
    optimizer.step()

    rl= loss.item()
print(f"Epoch {epochs+1}/{epochs}, Loss: {rl/len(trainloader):.4f}")


Epoch 20/19, Loss: 0.0003


In [27]:
correct=0
total=0
net.eval()
with torch.no_grad():
  for images, labels in testLoader:
    images,labels=images,labels
    outputs=net(images)
    _,predicted=torch.max(outputs,1)
    total+=labels.size(0)
    correct+=(predicted==labels).sum().item()
print(f"Test Accuracy: {100 * correct / total:.2f}%")

Test Accuracy: 70.66%
